# M0 Day01 실습 — 실전 사례: 업무자동화와 ESG 판단 모델

**목표**: Day00에서 배운 기법(역할 프롬프트·구조화된 출력·CoT)이 실제 업무에서 어떻게 쓰이는지 두 가지 실전 사례로 확인한다.
**구성**: Part 1 업무자동화(고장부품 결측치 자동 분류) → Part 2 ESG 판단 모델(CoT 유무 비교)

> **참고:** 이 노트북은 `OpenAI_API를_이용한_서비스_개발(24_12_15_강의용).ipynb`(raw `openai` SDK 기반 옛 강의자료)의 6~7절 실전 사례를 LangChain(`ChatPromptTemplate`)으로 다시 작성한 것이다. 원본은 `df`(pandas)에 의존했지만, 이 프로젝트엔 pandas 의존성이 없어 리스트(list[dict])로 바꿔 별도 데이터 파일 없이 실행되게 했다. ESG 사례도 원본의 실제 기업·인물 실명 기사 대신 가상 사례로 재구성했다.

## 0. 환경 준비

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()
print("준비 완료:", llm.model_name)

---

## Part 1. 업무자동화 — 고장부품 자동 분류

AS센터에 접수된 고장 설명(자유 텍스트)에서 원인 부품을 자동으로 분류해, 담당자가 일일이 채워 넣던 "고장부품" 필드의 결측치를 LLM으로 채운다. 사람이 직접 입력하던 반복 작업을 LLM 호출로 대체하는 가장 단순한 형태의 업무자동화다.

In [ ]:
tickets = [
    {"고장내용": "노트북 화면이 켜지지 않고 전원 버튼을 눌러도 반응이 없습니다.", "고장부품": "메인보드"},
    {"고장내용": "키보드 특정 자판(스페이스바)이 눌리지 않습니다.", "고장부품": "키보드"},
    {"고장내용": "충전은 되는데 배터리가 30분만에 방전됩니다.", "고장부품": None},
    {"고장내용": "스피커에서 지지직거리는 잡음이 계속 납니다.", "고장부품": None},
    {"고장내용": "USB 포트에 마우스를 꽂아도 인식이 안 됩니다.", "고장부품": "USB 포트"},
]

print("--- 원본 접수 목록 ---")
for t in tickets:
    print(t)

### 결측치만 골라 LLM으로 자동 분류

`고장부품`이 `None`인 항목만 골라, 상담원 역할 프롬프트로 부품 이름 한 단어만 뽑는다.

In [ ]:
# TODO: AS센터 상담원 역할로, 고장 설명을 읽고 원인 부품 이름만 한 단어로
#       답하도록(다른 말은 하지 않도록) 지시하는 시스템 프롬프트를 작성하세요
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", None),
    ("human", "{content}"),
])
classify_chain = classify_prompt | llm | parser

def classify_part(content):
    return classify_chain.invoke({"content": content}).strip()

print("--- 고장부품 결측치 자동 분류 ---")
for t in tickets:
    if t["고장부품"] is None:
        part = classify_part(t["고장내용"])
        t["고장부품"] = part
        print(f"{t['고장내용']} → {part}")

print()
print("--- 채워진 결과 ---")
for t in tickets:
    print(t)

---

## Part 2. ESG 판단 모델 — CoT 유무에 따른 점수 차이

같은 뉴스 기사를 지배구조(Governance) 관점에서 점수화할 때, "점수만 답하라"고 시킨 경우와 "근거를 먼저 쓰고 점수를 답하라"고 시킨 경우 결과가 어떻게 달라지는지 비교한다.

In [ ]:
esg_criteria = """당신은 ESG 평가 전문가입니다. 아래 뉴스를 읽고 지배구조(Governance) 관점에서 0~10점으로 평가하세요.

평가 기준:
- 경영진의 불법·비윤리적 행위 여부
- 정보 공개의 투명성
- 주주·투자자 보호 수준"""

news = ("(가상 사례) OO기업이 개인정보 유출 사고를 인지한 지 6시간 만에 홈페이지에 공지하고 "
        "금융당국에 자진 신고했다. 다만 유출 원인이 내부 직원의 관리 소홀로 밝혀졌고, "
        "재발 방지 대책은 아직 발표되지 않았다.")

# TODO: 근거 없이 점수만 숫자로 답하라는 지시문을 esg_criteria 뒤에 이어 붙이세요
no_cot_prompt = ChatPromptTemplate.from_messages([
    ("system", esg_criteria + None),
    ("human", "{news}"),
])
# TODO: 먼저 근거를 2~3문장으로 설명한 뒤 마지막 줄에 '점수: N' 형식으로 답하라는
#       지시문을 esg_criteria 뒤에 이어 붙이세요
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", esg_criteria + None),
    ("human", "{news}"),
])

print("--- 근거 없이 점수만 ---")
print((no_cot_prompt | llm | parser).invoke({"news": news}))

print()
print("--- 근거를 먼저 쓰게 한 경우 ---")
print((cot_prompt | llm | parser).invoke({"news": news}))

---

## 핵심 정리

- **업무자동화**: 반복적으로 사람이 채워 넣던 필드(분류·라벨링)를, 출력 형식을 한 단어로 좁힌 역할 프롬프트로 대체할 수 있다. 파싱이 복잡해질수록(JSON 등) 실패 지점이 늘어나므로, 가능하면 출력 자체를 단순하게 요구하는 쪽이 더 견고하다.
- **ESG 판단 모델**: 점수·등급처럼 정답이 하나로 정해지지 않는 판단형 과제에서도 CoT(근거를 먼저 쓰게 하기)는 결과에 실질적인 영향을 준다 — 다만 정답률처럼 "더 정확해졌다"고 단정할 수 없고, 근거의 타당성을 사람이 검토해야 한다.

**확인 질문**
1. Part 1에서 부품 이름을 JSON이 아니라 "한 단어만" 요구한 이유는 무엇인가?
2. Part 2에서 점수가 달라진 이유를, 근거 텍스트를 읽고 설명해보자.
3. 이 두 사례에 M1(Day03 Structured Output, Day04 Prompt Injection 방어)의 기법을 추가한다면 어디에 적용하고 싶은가?